# Molecular docking with AutoDock Vina

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yboulaamane/cadd-labs/blob/main/docking_autodock_vina.ipynb)

Docking predicts how a small molecule sits in a protein's pocket and scores how well it fits.
Here you'll dock a set of inhibitors into human monoamine oxidase B (MAO-B), the target of the
Parkinson's drugs selegiline, rasagiline and safinamide, using the crystal structure
[2V5Z](https://www.rcsb.org/structure/2V5Z) and [AutoDock Vina](https://vina.scripps.edu/).

**What you'll do**

- Prepare the receptor from the PDB, keeping the all-important FAD cofactor.
- Put the search box on the known binding site, taken from the crystal ligand.
- **Redock the crystal ligand and measure the RMSD**, the standard way to check a docking setup
  before you trust it.
- Dock the other compounds, rank them by score, and view the best pose in 3D.

Runtime: CPU is fine.

## Setup

RDKit, [Meeko](https://github.com/forlilab/Meeko) (ligand and cofactor preparation),
Open Babel (receptor protonation) and py3Dmol. Vina itself is the official static binary,
so the version does not depend on the Python version Colab happens to run.

In [ ]:
%pip install -q rdkit meeko openbabel-wheel py3Dmol

In [ ]:
import os

VINA_URL = "https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.7/vina_1.2.7_linux_x86_64"
if not os.path.exists("vina"):
    !wget -q "{VINA_URL}" -O vina && chmod +x vina
!./vina --version

In [ ]:
import subprocess

import numpy as np
import pandas as pd
import py3Dmol
import seaborn as sns
import matplotlib.pyplot as plt
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, PandasTools, rdMolAlign
from meeko import MoleculePreparation, PDBQTWriterLegacy, PDBQTMolecule, RDKitMolCreate

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({"figure.dpi": 120, "axes.titleweight": "bold", "savefig.bbox": "tight"})
RDLogger.DisableLog("rdApp.*")

## The structures come from the PDB file

A PDB file lists atoms and their coordinates but not their bond orders, so RDKit cannot
read a ligand from it directly. The helper below rebuilds each small molecule by matching
the atoms to a SMILES template. The SMILES for the two small molecules in 2V5Z (safinamide
and the FAD cofactor) are taken from the wwPDB chemical component dictionary.

In [ ]:
LIGAND_SMILES = {'SAG': 'C[C@H](NCc1ccc(OCc2cccc(F)c2)cc1)C(N)=O', 'FAD': 'Cc1cc2nc3c(=O)[nH]c(=O)nc-3n(CC(O)C(O)C(O)COP(=O)([O-])OP(=O)([O-])OCC3OC(n4cnc5c(N)ncnc54)C(O)C3O)c2cc1C'}

In [ ]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem


def read_pdb_lines(pdb_path):
    """Return (protein_lines, hetero_by_residue). Keeps the first altloc, drops waters."""
    protein, hetero = [], {}
    for line in open(pdb_path):
        record = line[:6].strip()
        if record not in ("ATOM", "HETATM"):
            continue
        if line[16] not in " A":                 # keep only the first alternate location
            continue
        resname = line[17:20].strip()
        if record == "ATOM":
            protein.append(line)
        elif resname != "HOH":                    # a small molecule, not a crystal water
            key = (resname, line[21], line[22:26].strip())
            hetero.setdefault(key, []).append(line)
    return protein, hetero


def residue_from_lines(lines, smiles):
    """Build an RDKit molecule from PDB atom lines, taking bond orders from `smiles`.

    PDB files store no bond orders. RDKit guesses bonds from distances; this function
    then sets their orders, charges and hydrogens from the SMILES template. Guessed bonds
    the template does not have (such as the spurious P-P bond RDKit adds across the two
    phosphates of FAD) are removed before matching.
    """
    block = "".join(l for l in lines if l[76:78].strip() != "H")
    guessed = Chem.RWMol(Chem.MolFromPDBBlock(block, removeHs=True, sanitize=False))
    guessed.UpdatePropertyCache(strict=False)
    Chem.FastFindRings(guessed)

    template = Chem.MolFromSmiles(smiles)
    skeleton = Chem.RWMol(template)
    for bond in skeleton.GetBonds():
        bond.SetBondType(Chem.BondType.SINGLE)
        bond.SetIsAromatic(False)
    for atom in skeleton.GetAtoms():
        atom.SetFormalCharge(0)
        atom.SetIsAromatic(False)
    skeleton.UpdatePropertyCache(strict=False)

    match = guessed.GetSubstructMatch(skeleton)
    if not match or len(match) != guessed.GetNumAtoms():
        raise ValueError("The atoms in the PDB residue do not match the SMILES given for it")
    keep = {frozenset((match[b.GetBeginAtomIdx()], match[b.GetEndAtomIdx()]))
            for b in template.GetBonds()}
    for bond in list(guessed.GetBonds()):
        if frozenset((bond.GetBeginAtomIdx(), bond.GetEndAtomIdx())) not in keep:
            guessed.RemoveBond(bond.GetBeginAtomIdx(), bond.GetEndAtomIdx())

    mol = AllChem.AssignBondOrdersFromTemplate(template, guessed.GetMol())
    Chem.AssignStereochemistryFrom3D(mol)
    return Chem.AddHs(mol, addCoords=True)

## Download and view 2V5Z

MAO-B is a homodimer; this notebook works with chain A. Each subunit holds one FAD
cofactor (yellow) and, in this structure, one safinamide molecule (green) in the site.

In [ ]:
!wget -q https://files.rcsb.org/download/2V5Z.pdb -O 2V5Z.pdb
protein_lines, hetero = read_pdb_lines("2V5Z.pdb")
chain_a = [line for line in protein_lines if line[21] == "A"]
print(f"{len(protein_lines)} protein atoms, chain A has {len(chain_a)}")
print("Small molecules:", sorted({(name, chain) for name, chain, _ in hetero}))

In [ ]:
view = py3Dmol.view(width=700, height=450)
view.addModel("".join(chain_a), "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
for resname, color in [("FAD", "yellow"), ("SAG", "green")]:
    lines = "".join("".join(v) for (r, c, _), v in hetero.items()
                    if r == resname and c == "A")
    view.addModel(lines, "pdb")
    view.setStyle({"model": -1}, {"stick": {"colorscheme": f"{color}Carbon"}})
view.zoomTo({"model": -1})
view.show()

## Prepare the receptor

The receptor is the protein plus its FAD cofactor, which lines the binding site and must
be present for docking. Open Babel adds hydrogens to the protein at pH 7.4 and writes it
in the PDBQT format Vina reads; FAD is prepared separately from its template and appended.

In [ ]:
def prepare_cofactor_pdbqt(lines, smiles, resname, resnum):
    mol = residue_from_lines(lines, smiles)
    setup = MoleculePreparation().prepare(mol)[0]
    pdbqt = PDBQTWriterLegacy.write_string(setup)[0]
    # Vina ignores residue names, but keeping them makes the file readable
    return [f"{l[:17]}{resname:>3} {l[21]}{resnum:>4}{l[26:]}"
            for l in pdbqt.splitlines() if l.startswith(("ATOM", "HETATM"))]


# protein of chain A, protonated by Open Babel
open("protein_A.pdb", "w").writelines(chain_a + ["END\n"])
subprocess.run(["obabel", "protein_A.pdb", "-xr", "-p", "7.4", "-O", "protein_A.pdbqt"],
               check=True, capture_output=True)
receptor = [l for l in open("protein_A.pdbqt").read().splitlines() if l.startswith(("ATOM", "HETATM"))]

fad_lines = next(v for (name, chain, _), v in hetero.items() if name == "FAD" and chain == "A")
receptor += prepare_cofactor_pdbqt(fad_lines, LIGAND_SMILES["FAD"], "FAD", 600)
open("receptor.pdbqt", "w").write("\n".join(receptor) + "\n")
print(f"Receptor: {len(receptor)} atoms (protein + FAD)")

## Place the search box

Vina searches inside a box. Centre it on the crystal ligand and add 5 Angstrom of padding
on each side, so the box covers the whole binding site.

In [ ]:
sag_lines = next(v for (name, chain, _), v in hetero.items() if name == "SAG" and chain == "A")
crystal_ligand = residue_from_lines(sag_lines, LIGAND_SMILES["SAG"])
heavy = np.array([crystal_ligand.GetConformer().GetAtomPosition(a.GetIdx())
                  for a in crystal_ligand.GetAtoms() if a.GetAtomicNum() > 1])
center = heavy.mean(axis=0)
box = np.ptp(heavy, axis=0) + 10.0
print("box centre (x, y, z):", center.round(2))
print("box size   (x, y, z):", box.round(1))

## Prepare and dock the compounds

Each compound is built from its SMILES, given a 3D conformer and written as PDBQT. The set
is the three MAO-B drugs plus a few natural coumarins; safinamide is included so its docked
pose can be compared with the crystal pose.

In [ ]:
compounds = {
    "safinamide": "C[C@H](NCc1ccc(OCc2cccc(F)c2)cc1)C(N)=O",
    "rasagiline": "C#CCN[C@@H]1CCc2ccccc21",
    "selegiline": "C#CCN(C)[C@H](C)Cc1ccccc1",
    "coumarin": "O=c1ccc2ccccc2o1",
    "umbelliferone": "O=c1ccc2ccc(O)cc2o1",
    "scopoletin": "COc1cc2ccc(=O)oc2cc1O",
    "osthole": "COc1ccc2ccc(=O)oc2c1CC=C(C)C",
}


def ligand_pdbqt(smiles, path):
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    AllChem.EmbedMolecule(mol, randomSeed=42)
    AllChem.MMFFOptimizeMolecule(mol)
    setup = MoleculePreparation().prepare(mol)[0]
    open(path, "w").write(PDBQTWriterLegacy.write_string(setup)[0])


def dock(smiles, name, exhaustiveness=16):
    ligand_pdbqt(smiles, f"{name}.pdbqt")
    subprocess.run(["./vina", "--receptor", "receptor.pdbqt", "--ligand", f"{name}.pdbqt",
                    "--center_x", f"{center[0]:.3f}", "--center_y", f"{center[1]:.3f}",
                    "--center_z", f"{center[2]:.3f}", "--size_x", f"{box[0]:.1f}",
                    "--size_y", f"{box[1]:.1f}", "--size_z", f"{box[2]:.1f}", "--seed", "42",
                    "--exhaustiveness", str(exhaustiveness), "--out", f"{name}_out.pdbqt"],
                   check=True, capture_output=True, text=True)
    poses = PDBQTMolecule.from_file(f"{name}_out.pdbqt", skip_typing=True)
    best = RDKitMolCreate.from_pdbqt_mol(poses)[0]
    return best, [pose.score for pose in poses]

## Redocking: does the setup reproduce the crystal pose?

Before trusting any score, dock the crystal ligand back in and measure the RMSD between the
top docked pose and the crystal pose. Below about 2 Angstrom is the usual sign that the box
and receptor are set up well enough to dock related compounds.

In [ ]:
redock, redock_scores = dock(compounds["safinamide"], "redock")
rmsd = [rdMolAlign.CalcRMS(Chem.RemoveHs(redock), Chem.RemoveHs(crystal_ligand), prbId=c.GetId())
        for c in redock.GetConformers()]
print(f"Top pose score: {redock_scores[0]:.2f} kcal/mol, RMSD to crystal: {rmsd[0]:.2f} Angstrom")
print(f"Best RMSD among the {len(rmsd)} poses: {min(rmsd):.2f} Angstrom")

In [ ]:
view = py3Dmol.view(width=700, height=450)
view.addModel(Chem.MolToMolBlock(Chem.RemoveHs(crystal_ligand)), "mol")
view.setStyle({"model": -1}, {"stick": {"colorscheme": "greenCarbon"}})
view.addModel(Chem.MolToMolBlock(Chem.RemoveHs(redock), confId=0), "mol")
view.setStyle({"model": -1}, {"stick": {"colorscheme": "magentaCarbon"}})
view.zoomTo()
view.show()   # green: crystal pose, magenta: top docked pose

## Dock the set and compare scores

Vina's score is an estimate of the binding free energy in kcal/mol, so **more negative is a
better predicted binder**. The scores are useful for ranking compounds against the same
target; they are not accurate absolute affinities.

In [ ]:
results = []
for name, smiles in compounds.items():
    _, scores = dock(smiles, name)
    results.append({"compound": name, "smiles": smiles, "vina_score": scores[0]})

table = pd.DataFrame(results).sort_values("vina_score").reset_index(drop=True)
PandasTools.AddMoleculeColumnToFrame(table, "smiles", "structure")
table[["compound", "vina_score"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
colors = ["#d1495b" if c == "safinamide" else "#4e79a7" for c in table["compound"]]
ax.barh(table["compound"], table["vina_score"], color=colors)
ax.set(xlabel="Vina score (kcal/mol, more negative is better)", title="Predicted binding to MAO-B")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Takeaways

- More negative is a better *predicted* binder, but a Vina score is a ranking tool, not a measured
  affinity. Use it to prioritise which compounds to test, not to quote a Kd.
- The redocking RMSD is your trust check: if the crystal pose comes back below ~2 Angstrom, the
  setup is sound. Always do it before reading the scores.
- The receptor is rigid here, which is the biggest simplification. Real side chains move, and
  MAO-B has a gating residue (Ile199) that opens between its two cavities.
- To screen a whole library, run this same loop over a file of SMILES and keep the top scores.
  [VinaScreen](https://github.com/yboulaamane/VinaScreen) is a ready-made batch version.